In [1]:
# DASK client set

import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

# get path for path changes in Jupyter notebook: File - Open from Path - insert relative_path
notebook_path = os.path.abspath(".")
_, _, relative_path = notebook_path.partition('/all/')
relative_path = '/all/' + relative_path
relative_path

'/all/Model/CESM2/Earth_System_Predictability/DIC'

# Load modules

In [3]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.ticker as mticker
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.interpolate import griddata
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp


In [4]:
import xcesm
import gsw

In [5]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_NWP_preprocessing import CESM2_NWP_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

# Variable configuration

In [6]:

cfgs = {}
# varlist = ["DIC", "TEMP", "SALT"]
# varlist = ["DIC_ALT_CO2"]
varlist = ["TEMP"]

for varname in varlist:
    cfg = CESM2_NWP_config()
    cfg.year_s = 1954
    cfg.year_e = 2020
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')

# Read dataset

In [7]:
# lat_range = slice(23, 38)
# lon_range = slice(120, 180)
lat_range = slice(19, 40)
lon_range = slice(120, 180)

In [8]:
# define preprocessing function

# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]
# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'dz', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]


# exceptcv = [
#     'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
# ] + [cfg.var for cfg in cfgs.values()]
exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 
] + [cfg.var for cfg in cfgs.values()]


def process_coords_3d(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        # ds = ds.isel(z_t=slice(0, 39)) # ~39 layer (1000m)
        # ds = (ds.isel(z_t=slice(1, 39)) * ds.dz).sum(dim='z_t') / ds.dz.sum(dim='z_t')
        return ds
    else:
        return ds.set_coords(coord_vars)



def process_coords_3d_LE(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """
    Preprocessor function for CESM POP-style datasets.
    - Normalizes vertical coordinate: if z_t or z_t_2 exists, rename to 'depth'.
    - Replaces its values with z_t_new for consistency.
    - Optionally drops unnecessary coordinate variables for faster concatenation.
    """
    z_t_new = np.array([5.0000000e+00, 1.5000000e+01, 2.5000000e+01, 3.5000000e+01,
       4.5000000e+01, 5.5000000e+01, 6.5000000e+01, 7.5000000e+01,
       8.5000000e+01, 9.5000000e+01, 1.0500000e+02, 1.1500000e+02,
       1.2500000e+02, 1.3500000e+02, 1.4500000e+02, 1.5500000e+02,
       1.6509839e+02, 1.7547903e+02, 1.8629126e+02, 1.9766026e+02,
       2.0971138e+02, 2.2257828e+02, 2.3640883e+02, 2.5137015e+02,
       2.6765421e+02, 2.8548364e+02, 3.0511920e+02, 3.2686798e+02,
       3.5109348e+02, 3.7822760e+02, 4.0878464e+02, 4.4337769e+02,
       4.8273669e+02, 5.2772797e+02, 5.7937286e+02, 6.3886261e+02,
       7.0756329e+02, 7.8700250e+02, 8.7882520e+02, 9.8470581e+02,
       1.1062042e+03, 1.2445669e+03, 1.4004972e+03, 1.5739464e+03,
       1.7640033e+03, 1.9689442e+03, 2.1864565e+03, 2.4139714e+03,
       2.6490012e+03, 2.8893845e+03, 3.1334045e+03, 3.3797935e+03,
       3.6276702e+03, 3.8764519e+03, 4.1257681e+03, 4.3753926e+03,
       4.6251904e+03, 4.8750835e+03, 5.1250278e+03, 5.3750000e+03])
    
    # ------------------------------------------------------
    # 1️⃣ Normalize vertical coordinate name
    # ------------------------------------------------------
    if "z_t_2" in ds.dims:
        ds = ds.rename({"z_t_2": "depth"})
    elif "z_t" in ds.dims:
        ds = ds.rename({"z_t": "depth"})
    else:
        print("[Warning] No vertical coordinate (z_t or z_t_2) found — skipped.")
        return ds

    # Drop any leftover z_t/z_t_2 coordinate variable if it exists
    ds = ds.drop_vars(["z_t", "z_t_2"], errors="ignore")

    # ------------------------------------------------------
    # 2️⃣ Replace coordinate values with z_t_new
    # ------------------------------------------------------
    if "depth" in ds.coords:
        if len(ds["depth"]) == len(z_t_new):
            ds = ds.assign_coords(depth=z_t_new)
        else:
            print(f"[Warning] depth length mismatch: {len(ds['depth'])} vs {len(z_t_new)}")
    else:
        print("[Warning] depth coordinate missing after renaming.")

    # ------------------------------------------------------
    # 3️⃣ Clean up coordinate references inside variable attributes
    # ------------------------------------------------------
    for v in ds.data_vars:
        if "coordinates" in ds[v].attrs:
            ds[v].attrs["coordinates"] = (
                ds[v].attrs["coordinates"]
                .replace("z_t_2", "depth")
                .replace("z_t", "depth")
            )

    # ------------------------------------------------------
    # 4️⃣ Drop unnecessary coordinate variables and slice time
    # ------------------------------------------------------
    coord_vars = []
    for v in np.array(ds.coords):
        if v not in except_coord_vars:
            coord_vars.append(v)
    for v in np.array(ds.data_vars):
        if v not in except_coord_vars:
            coord_vars.append(v)

    if drop:
        ds = ds.drop(coord_vars, errors="ignore")
        ds = ds.sel(time=slice(sd, ed))
    else:
        ds = ds.set_coords(coord_vars)

    return ds



# exceptcv=['time','lon','lat','lev', 'TLONG', 'TLAT', 'z_t', cfg_var_DIC.var, cfg_var_Li_diat.var, cfg_var_Li_sp.var,
#          cfg_var_Vi_diat_Fe.var, cfg_var_Vi_diat_N.var, cfg_var_Vi_diat_P.var, cfg_var_Vi_diat_SiO3.var,
#          cfg_var_Vi_sp_Fe.var, cfg_var_Vi_sp_N.var, cfg_var_Vi_sp_P.var,
#          cfg_var_NPP_diat.var, cfg_var_NPP_sp.var]
# def process_coords(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
#     """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
#     coord_vars = []
#     for v in np.array(ds.coords) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     for v in np.array(ds.data_vars) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
    
#     if drop:
#         ds= ds.drop(coord_vars)
#         ds= ds.sel(time=slice(sd, ed))
#         return ds
#     else:
#         return ds.set_coords(coord_vars)

start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)


# ds = ds.isel(lev=slice(1, 11))

In [15]:
import os
import glob

BASE_RGD = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn"

vars = varlist
file_list_assm = {}

for var in vars:
    print(f"\n===== Collecting regridded files for {var} =====")

    var_dir = os.path.join(BASE_RGD, var)

    if not os.path.isdir(var_dir):
        print(f"Directory not found: {var_dir}")
        continue

    # Detect ensemble directories automatically
    ens_dirs = sorted(
        [
            d for d in os.listdir(var_dir)
            if os.path.isdir(os.path.join(var_dir, d))
        ]
    )

    print(f"Detected ensembles: {ens_dirs}")

    rgd_file_list = []

    for ens in ens_dirs:
        ens_path = os.path.join(var_dir, ens)

        # Collect all regridded files regardless of time segmentation
        files = sorted(
            glob.glob(os.path.join(ens_path, "regridded_*.nc"))
        )

        # print(f"\n--- Ensemble {ens} ---")
        # print(f"Number of files: {len(files)}")

        # for f in files:
        #     print(f)

        rgd_file_list.append(files)

    file_list_assm[var] = rgd_file_list


    
#     print(f"\n===== Processing {var} =====")

#     cfg_var = cfgs[var]
#     cfg_var.var = var
# # ================= ODA =================
#     t0 = time.time()
#     cfg_var.ODA_path_load(cfg_var.var)
#     cfg_var.ODA_ds_rgd = xr.open_mfdataset(
#         file_list_assm[var][10:20],
#         chunks={'time': 12},
#         combine='nested',
#         concat_dim=[[*cfg_var.ODA_ensembles][10:20], 'time'],
#         parallel=True,
#         preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
#         decode_cf=True,
#         decode_times=True,
#     )
#     cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.rename({"concat_dim": "ens_ODA"})
#     cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.sortby("time")
#     print(f"  ODA read: {time.time() - t0:.1f} s")




===== Collecting regridded files for TEMP =====
Detected ensembles: ['en4.2_ba-10p1', 'en4.2_ba-10p2', 'en4.2_ba-10p3', 'en4.2_ba-10p4', 'en4.2_ba-10p5', 'en4.2_ba-20p1', 'en4.2_ba-20p2', 'en4.2_ba-20p3', 'en4.2_ba-20p4', 'en4.2_ba-20p5', 'projdv7.3_ba-10p1', 'projdv7.3_ba-10p2', 'projdv7.3_ba-10p3', 'projdv7.3_ba-10p4', 'projdv7.3_ba-10p5', 'projdv7.3_ba-20p1', 'projdv7.3_ba-20p2', 'projdv7.3_ba-20p3', 'projdv7.3_ba-20p4', 'projdv7.3_ba-20p5']

===== Processing TEMP =====
  ODA read: 3.1 s


In [25]:
import xarray as xr
import numpy as np
import os

# rgd_file_list = cfgs["TEMP"].rgd_file_list  (이미 존재한다고 가정)

def compute_MTD(temp, T_iso=12.0):
    """
    temp dims: (..., z_t, lat, lon) or (time,z_t,lat,lon)
    z_t in cm
    """

    z = temp["z_t"]

    # where temperature drops below isotherm
    below = temp < T_iso

    # first level below isotherm
    k1 = below.argmax("z_t")

    # check crossing
    has_above = (temp >= T_iso).any("z_t")
    has_below = below.any("z_t")
    crosses = has_above & has_below & (k1 > 0)

    # upper level index
    k0 = (k1 - 1)

    # broadcast z to temp shape
    z3 = z.broadcast_like(temp)

    # extract bracket values
    t0 = temp.isel(z_t=k0)
    t1 = temp.isel(z_t=k1)
    z0 = z3.isel(z_t=k0)
    z1 = z3.isel(z_t=k1)

    # interpolation
    frac = (T_iso - t0) / (t1 - t0)
    MTD = z0 + frac * (z1 - z0)

    # mask non-crossing columns
    MTD = MTD.where(crosses)

    MTD.name = "MTD"
    MTD.attrs["long_name"] = f"Depth of {T_iso}C isotherm"
    MTD.attrs["units"] = "cm"

    return MTD



for ens_files in rgd_file_list:          # each ensemble
    for temp_file in ens_files:          # each time chunk

        print("TEMP:", temp_file)

        ds = xr.open_dataset(temp_file)
        temp = ds["TEMP"]

        MTD = compute_MTD(temp, T_iso=12.0)

        # build output path (TEMP → MTD)
        out_file = temp_file.replace("/TEMP/", "/MTD/")
        out_file = out_file.replace(".TEMP.", ".MTD.")
        out_file = out_file.replace("_TEMP_", "_MTD_")

        out_dir = os.path.dirname(out_file)
        os.makedirs(out_dir, exist_ok=True)

        encoding = {
            "MTD": {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32"
            }
        }

        MTD.to_netcdf(out_file, encoding=encoding)

        ds.close()
        del ds, temp, MTD

        print("MTD :", out_file)


TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/TEMP/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.TEMP.195101-195512.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/MTD/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.MTD.195101-195512.nc
TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/TEMP/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.TEMP.195601-196012.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/MTD/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.MTD.195601-196012.nc
TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/TEMP/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.TEMP.196101-196512.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn/MTD/en4.2_ba-10p1/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.en4.2_ba-10p1.pop.h.MTD.196101-19

In [9]:
import os
import glob
import xarray as xr
import time

BASE_RGD = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn"

vars = varlist
file_list_WDA = {}

for var in vars:
    print(f"\n===== Collecting WDA regridded files for {var} =====")

    var_dir = os.path.join(BASE_RGD, var)

    if not os.path.isdir(var_dir):
        print(f"Directory not found: {var_dir}")
        continue

    # WDA: files directly under var_dir (no ensemble)
    files = sorted(
        glob.glob(os.path.join(var_dir, "regridded_*.nc"))
    )

    print(f"Number of files: {len(files)}")

    file_list_WDA[var] = files



===== Collecting WDA regridded files for TEMP =====
Number of files: 14


In [11]:
import xarray as xr
import numpy as np
import os

# ============================================================
# INPUT: already collected WDA regridded TEMP files
# ============================================================
file_list = file_list_WDA["TEMP"]   # ← 핵심 변경 (WDA list)

# ============================================================
# MTD function
# ============================================================
def compute_MTD(temp, T_iso=12.0):
    """
    temp dims: (..., z_t, lat, lon)
    z_t in cm
    """
    z = temp["z_t"]

    below = temp < T_iso
    k1 = below.argmax("z_t")

    has_above = (temp >= T_iso).any("z_t")
    has_below = below.any("z_t")
    crosses = has_above & has_below & (k1 > 0)

    k0 = (k1 - 1)

    z3 = z.broadcast_like(temp)

    t0 = temp.isel(z_t=k0)
    t1 = temp.isel(z_t=k1)
    z0 = z3.isel(z_t=k0)
    z1 = z3.isel(z_t=k1)

    frac = (T_iso - t0) / (t1 - t0)
    MTD = z0 + frac * (z1 - z0)

    MTD = MTD.where(crosses)

    MTD.name = "MTD"
    MTD.attrs["long_name"] = f"Depth of {T_iso}C isotherm"
    MTD.attrs["units"] = "cm"

    return MTD


# ============================================================
# WDA loop (no ensemble)
# ============================================================
for temp_file in file_list:

    print("TEMP:", temp_file)

    ds = xr.open_dataset(temp_file)
    temp = ds["TEMP"]

    MTD = compute_MTD(temp, T_iso=12.0)

    # TEMP → MTD 경로 변환
    out_file = temp_file.replace("/TEMP/", "/MTD/")
    out_file = out_file.replace(".TEMP.", ".MTD.")
    out_file = out_file.replace("_TEMP_", "_MTD_")

    out_dir = os.path.dirname(out_file)
    os.makedirs(out_dir, exist_ok=True)

    encoding = {
        "MTD": {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }
    }

    MTD.to_netcdf(out_file, encoding=encoding)

    ds.close()
    del ds, temp, MTD

    print("MTD :", out_file)


TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/TEMP/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.TEMP.195501-195912.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/MTD/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.MTD.195501-195912.nc
TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/TEMP/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.TEMP.196001-196412.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/MTD/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.MTD.196001-196412.nc
TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/TEMP/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.TEMP.196501-196912.nc
MTD : /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn/MTD/regridded_NWP_b.e21.BHISTsmbb.f09_g17.assm.projdv7.3_ba-20p1.pop.h.MTD.196501-196912.nc
TEMP: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded